# Balanced Accuracy Threshold Tuning

The leaderboard metric is balanced accuracy, so this notebook optimizes average recall across classes instead of plain accuracy. It trains only on the train split, tunes probability multipliers on validation, then retrains on train+validation and applies the selected rule to test.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import balanced_accuracy_score, accuracy_score, classification_report, confusion_matrix, recall_score
from sklearn.preprocessing import LabelEncoder

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 160)

RANDOM_STATE = 42
ID_COL = "id"
TARGET_COL = "health_condition"

## Load Numeric Feature Splits

In [2]:
train_df = pd.read_csv("data/train_split_features_numeric.csv")
val_df = pd.read_csv("data/val_split_features_numeric.csv")
test_df = pd.read_csv("data/test_features_numeric.csv")
sample_submission = pd.read_csv("data/sample_submission.csv")

feature_cols = [col for col in train_df.columns if col not in [ID_COL, TARGET_COL]]
X_train = train_df[feature_cols].astype("float32")
X_val = val_df[feature_cols].astype("float32")
X_test = test_df[feature_cols].astype("float32")
y_train_raw = train_df[TARGET_COL]
y_val_raw = val_df[TARGET_COL]

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw)
y_val = label_encoder.transform(y_val_raw)
class_names = label_encoder.classes_.tolist()
class_id = {name: int(label_encoder.transform([name])[0]) for name in class_names}

print("train:", train_df.shape, "val:", val_df.shape, "test:", test_df.shape)
print("classes:", class_id)
print("validation target distribution (%):")
print(y_val_raw.value_counts(normalize=True).mul(100).round(2))

train: (552070, 73) val: (138018, 73) test: (295753, 72)
classes: {'at-risk': 0, 'fit': 1, 'unhealthy': 2}
validation target distribution (%):
at-risk      85.87
unhealthy     8.36
fit           5.77
Name: health_condition, dtype: float64


## Train Balanced Boosted Model On Train Split

In [3]:
model = HistGradientBoostingClassifier(
    loss="log_loss",
    learning_rate=0.06,
    max_iter=500,
    max_leaf_nodes=31,
    l2_regularization=0.01,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=25,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    verbose=0,
)
model.fit(X_train, y_train)
print("selected iterations:", int(model.n_iter_))

selected iterations: 123


## Baseline Validation Metrics

In [4]:
val_proba = model.predict_proba(X_val)
base_pred = val_proba.argmax(axis=1)
base_labels = label_encoder.inverse_transform(base_pred)

print("accuracy:", round(accuracy_score(y_val, base_pred), 6))
print("balanced accuracy:", round(balanced_accuracy_score(y_val, base_pred), 6))
print("per-class recall:")
for name, rec in zip(class_names, recall_score(y_val, base_pred, average=None, labels=np.arange(len(class_names)))):
    print(f"  {name}: {rec:.6f}")
print("prediction distribution (%):")
print(pd.Series(base_labels).value_counts(normalize=True).mul(100).round(2))
print("classification report:")
print(classification_report(y_val, base_pred, target_names=class_names, digits=4))

accuracy: 0.882624
balanced accuracy: 0.910257
per-class recall:
  at-risk: 0.874966
  fit: 0.921115
  unhealthy: 0.934690
prediction distribution (%):
at-risk      76.07
unhealthy    13.65
fit          10.28
dtype: float64
classification report:
              precision    recall  f1-score   support

     at-risk     0.9876    0.8750    0.9279    118512
         fit     0.5169    0.9211    0.6622      7961
   unhealthy     0.5729    0.9347    0.7104     11545

    accuracy                         0.8826    138018
   macro avg     0.6925    0.9103    0.7668    138018
weighted avg     0.9258    0.8826    0.8944    138018



## Tune Class Probability Multipliers On Validation

For each row, the prediction becomes `argmax(probability * class_multiplier)`. Multipliers are learned only from validation and then reused on test.

In [5]:
at_idx = class_id["at-risk"]
fit_idx = class_id["fit"]
unhealthy_idx = class_id["unhealthy"]

fit_grid = np.round(np.arange(0.60, 1.61, 0.05), 2)
unhealthy_grid = np.round(np.arange(0.60, 1.61, 0.05), 2)

rows = []
for fit_mult in fit_grid:
    for unhealthy_mult in unhealthy_grid:
        multipliers = np.ones(len(class_names), dtype="float32")
        multipliers[fit_idx] = fit_mult
        multipliers[unhealthy_idx] = unhealthy_mult
        pred = (val_proba * multipliers).argmax(axis=1)
        recalls = recall_score(y_val, pred, average=None, labels=np.arange(len(class_names)))
        pred_labels = label_encoder.inverse_transform(pred)
        dist = pd.Series(pred_labels).value_counts(normalize=True).mul(100).to_dict()
        rows.append({
            "fit_mult": fit_mult,
            "unhealthy_mult": unhealthy_mult,
            "balanced_accuracy": balanced_accuracy_score(y_val, pred),
            "accuracy": accuracy_score(y_val, pred),
            "recall_at_risk": recalls[at_idx],
            "recall_fit": recalls[fit_idx],
            "recall_unhealthy": recalls[unhealthy_idx],
            "pred_at_risk_pct": dist.get("at-risk", 0),
            "pred_fit_pct": dist.get("fit", 0),
            "pred_unhealthy_pct": dist.get("unhealthy", 0),
        })

tuning = pd.DataFrame(rows).sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
tuning.head(20)

,fit_mult,unhealthy_mult,balanced_accuracy,accuracy,recall_at_risk,recall_fit,recall_unhealthy,pred_at_risk_pct,pred_fit_pct,pred_unhealthy_pct
0,1.15,1.05,0.910733,0.872241,0.861769,0.931667,0.938761,74.841687,11.157240,14.001072
1,1.20,1.05,0.910675,0.869879,0.858833,0.934430,0.938761,74.573606,11.425321,14.001072
2,1.10,1.05,0.910672,0.874299,0.864351,0.928903,0.938761,75.081511,10.917417,14.001072
3,1.15,1.00,0.910613,0.875089,0.865482,0.931667,0.934690,75.195989,11.157240,13.646771
4,1.15,0.95,0.910601,0.878146,0.869431,0.931667,0.930706,75.570578,11.157240,13.272182
5,1.05,1.05,0.910567,0.877146,0.867929,0.925009,0.938761,75.411178,10.587749,14.001072
6,1.20,1.00,0.910555,0.872727,0.862546,0.934430,0.934690,74.927908,11.425321,13.646771
7,1.10,1.00,0.910553,0.877146,0.868064,0.928903,0.934690,75.435813,10.917417,13.646771
8,1.20,0.95,0.910544,0.875784,0.866495,0.934430,0.930706,75.302497,11.425321,13.272182
9,1.10,0.95,0.910541,0.880204,0.872013,0.928903,0.930706,75.810402,10.917417,13.272182


## Select Practical Variants

In [6]:
baseline_row = {
    "variant": "baseline_balanced",
    "fit_mult": 1.0,
    "unhealthy_mult": 1.0,
}

best = tuning.iloc[0]

# Pick a conservative option: within 0.001 balanced accuracy of best, but closest to baseline multipliers.
near_best = tuning[tuning["balanced_accuracy"] >= best["balanced_accuracy"] - 0.001].copy()
near_best["distance_from_1"] = (near_best["fit_mult"] - 1).abs() + (near_best["unhealthy_mult"] - 1).abs()
conservative = near_best.sort_values(["distance_from_1", "balanced_accuracy"], ascending=[True, False]).iloc[0]

# Pick minority-leaning option: within 0.002 of best, with stronger combined minority recall.
minority_pool = tuning[tuning["balanced_accuracy"] >= best["balanced_accuracy"] - 0.002].copy()
minority_pool["minority_recall"] = (minority_pool["recall_fit"] + minority_pool["recall_unhealthy"]) / 2
minority = minority_pool.sort_values(["minority_recall", "balanced_accuracy"], ascending=False).iloc[0]

selected = pd.DataFrame([
    {"variant": "best_bacc", "fit_mult": float(best.fit_mult), "unhealthy_mult": float(best.unhealthy_mult)},
    {"variant": "conservative_bacc", "fit_mult": float(conservative.fit_mult), "unhealthy_mult": float(conservative.unhealthy_mult)},
    {"variant": "minority_recall_bacc", "fit_mult": float(minority.fit_mult), "unhealthy_mult": float(minority.unhealthy_mult)},
    baseline_row,
])

selected = selected.drop_duplicates(subset=["fit_mult", "unhealthy_mult"]).reset_index(drop=True)
selected.merge(tuning, on=["fit_mult", "unhealthy_mult"], how="left")

,variant,fit_mult,unhealthy_mult,balanced_accuracy,accuracy,recall_at_risk,recall_fit,recall_unhealthy,pred_at_risk_pct,pred_fit_pct,pred_unhealthy_pct
0,best_bacc,1.15,1.05,0.910733,0.872241,0.861769,0.931667,0.938761,74.841687,11.157240,14.001072
1,conservative_bacc,1.00,1.00,0.910257,0.882624,0.874966,0.921115,0.934690,76.074860,10.278369,13.646771
2,minority_recall_bacc,1.25,1.50,0.908904,0.848839,0.832321,0.936315,0.958077,72.114507,11.655002,16.230492


## Retrain On Train + Validation, Apply Tuned Rules To Test

In [7]:
full_df = pd.concat([train_df, val_df], ignore_index=True)
X_full = full_df[feature_cols].astype("float32")
y_full = label_encoder.transform(full_df[TARGET_COL])

final_model = HistGradientBoostingClassifier(
    loss="log_loss",
    learning_rate=0.06,
    max_iter=int(model.n_iter_),
    max_leaf_nodes=31,
    l2_regularization=0.01,
    early_stopping=False,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    verbose=0,
)
final_model.fit(X_full, y_full)
test_proba = final_model.predict_proba(X_test)

submission_rows = []
for _, row in selected.iterrows():
    multipliers = np.ones(len(class_names), dtype="float32")
    multipliers[fit_idx] = row["fit_mult"]
    multipliers[unhealthy_idx] = row["unhealthy_mult"]

    test_pred = (test_proba * multipliers).argmax(axis=1)
    test_labels = label_encoder.inverse_transform(test_pred)

    submission = sample_submission.copy()
    submission[ID_COL] = test_df[ID_COL].values
    submission[TARGET_COL] = test_labels

    output_path = Path(f"data/submission_bacc_threshold_{row['variant']}.csv")
    submission.to_csv(output_path, index=False)

    dist = submission[TARGET_COL].value_counts(normalize=True).mul(100).to_dict()
    submission_rows.append({
        "variant": row["variant"],
        "path": str(output_path),
        "fit_mult": row["fit_mult"],
        "unhealthy_mult": row["unhealthy_mult"],
        "test_at_risk_pct": dist.get("at-risk", 0),
        "test_fit_pct": dist.get("fit", 0),
        "test_unhealthy_pct": dist.get("unhealthy", 0),
    })

submission_report = pd.DataFrame(submission_rows)
submission_report

,variant,path,fit_mult,unhealthy_mult,test_at_risk_pct,test_fit_pct,test_unhealthy_pct
0,best_bacc,data/submission_bacc_threshold_best_bacc.csv,1.15,1.05,74.372872,11.438937,14.188191
1,conservative_bacc,data/submission_bacc_threshold_conservative_bacc.csv,1.00,1.00,75.610391,10.526351,13.863258
2,minority_recall_bacc,data/submission_bacc_threshold_minority_recall_bacc.csv,1.25,1.50,71.957681,11.905205,16.137114


## Compare With Existing Public Candidates

In [8]:
compare_paths = [
    "data/submission_boosted_balanced.csv",
    "data/submission_catboost_balanced_fast.csv",
    "data/submission_ensemble_catboost_nn_agree_only.csv",
    "data/submission_ensemble_minority_rescue_catboost_nn.csv",
] + submission_report["path"].tolist()

rows = []
base_submission = pd.read_csv("data/submission_boosted_balanced.csv")
for path in compare_paths:
    p = Path(path)
    if not p.exists():
        continue
    sub = pd.read_csv(p)
    dist = sub[TARGET_COL].value_counts(normalize=True).mul(100).to_dict()
    rows.append({
        "file": p.name,
        "changed_vs_boosted": int((sub[TARGET_COL] != base_submission[TARGET_COL]).sum()),
        "changed_pct": round((sub[TARGET_COL] != base_submission[TARGET_COL]).mean() * 100, 2),
        "at-risk": dist.get("at-risk", 0),
        "fit": dist.get("fit", 0),
        "unhealthy": dist.get("unhealthy", 0),
    })

pd.DataFrame(rows)

,file,changed_vs_boosted,changed_pct,at-risk,fit,unhealthy
0,submission_boosted_balanced.csv,0,0.00,75.610391,10.526351,13.863258
1,submission_catboost_balanced_fast.csv,6613,2.24,75.590104,10.391780,14.018116
2,submission_ensemble_catboost_nn_agree_only.csv,3955,1.34,75.206676,10.760330,14.032994
3,submission_ensemble_minority_rescue_catboost_nn.csv,2559,0.87,74.745142,10.971655,14.283203
4,submission_bacc_threshold_best_bacc.csv,3660,1.24,74.372872,11.438937,14.188191
5,submission_bacc_threshold_conservative_bacc.csv,0,0.00,75.610391,10.526351,13.863258
6,submission_bacc_threshold_minority_recall_bacc.csv,10803,3.65,71.957681,11.905205,16.137114
